In [31]:
%matplotlib inline
import platform
from importlib.metadata import version
from abc import ABC, abstractmethod
from qiskit.quantum_info import random_unitary,Operator


import matplotlib.pyplot as plt
import math
import numpy as np
from qiskit.circuit.library import DiagonalGate
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Operator, Statevector, partial_trace
from oracle_conversions import*
ATOL = 1e-10
plt.rcParams.update(
    {
        "figure.figsize": (8, 3.8),
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Python", platform.python_version())
for package in ("qiskit", "numpy", "matplotlib"):
    print(package, version(package))

Python 3.12.3
qiskit 2.5.2
numpy 2.5.4
matplotlib 3.11.2


## Four Types of Oracles

A quantum computer can store data in a few different ways: as probabilities, as phases, as probabilities, or as basis states.

One goal in understanding quantum computation is to understand how to convert between different representations of data.

[Source](https://arxiv.org/pdf/1711.00465), see Definition 8.

### Function Oracles
A function oracle is a way to represent functions using quantum circuits.

Let $f:\mathbb{F}_2^n\to [0,1]$ be a function. A <i>function oracle</i> is a unitary operator $U_f$ of dimension $2^{n+k}\times 2^{n+k}$ (for some $k$, the number of auxiliary qubits) that has the property that for every $x\in \mathbb{F}_2^n$,
\begin{align*}
U_f\ket{x}\ket{{\bf 0}} = \ket{x}\otimes (\sqrt{f(x)} \ket{good(x)}\ket{1} + \sqrt{1-f(x)}\ket{bad(x)}\ket{0})
\end{align*}

Given a function oracle $U_f$, we can extract the value $f(x)$ by repeatedly preparing the state $\ket{x}\ket{{\bf 0}}$, applying $U_f$, and collecting outcome statistics of the last qubit, or we can apply amplitude estimation.

### Phase Oracles
A phase oracle is another way to represent functions using quantum circuits.

Let $f:\mathbb{F}_2^n\to [0,1]$ be a function. A <i>phase oracle</i> is a unitary operator $O_f$ of dimension $2^{n+k}\times 2^{n+k}$ (for some $k$, the number of auxiliary qubits) that has the property that for every $x\in \mathbb{F}_2^n$,

\begin{align*}
O_f \ket{x}\ket{{\bf 0}} = e^{if(x)}\ket{x}\ket{{\bf 0}}
\end{align*}

Given a phase oracle $O_f$, we can extract the value $f(x)$ by phase estimation.

### Probability Oracles
A probability oracle is a way to represent probability distributions using quantum circuits.

Let $p:\mathbb{F}_2^n \to [0,1]$ be a probability distribution. A <i>probability oracle</i> is a unitary operator $H_p$ of dimension $2^{n+k}\times 2^{n+k}$ (for some $k$, the number of auxiliary qubits) that has the property that

\begin{align*}
H_p \ket{{\bf 0}} = \sum_{x\in \mathbb{F}_2^n} \sqrt{p(x)}\ket{x}\ket{\psi(x)}
\end{align*}

where $\ket{\psi(x)}$ is a normalized state. In other words, applying $H_p$ to the computational zero state and measuring the first $n$ qubits is the same as sampling from $p$.


### Binary Oracles
A binary oracle is a way to represent binary functions in basis states.

Let $f:\mathbb{F}_2^n \to \mathbb{F_2}^d$. A <i>binary oracle</i> is a unitary operator $B_f$ of dimension $2^{n+d+k} \times 2^{n+d+k}$ (for some $k$, the number of auxillary qubits) that has the property that

\begin{align*}
B_f \ket{x}\ket{y}\ket{{\bf}} = \ket{x}\ket{f(x)\oplus y}\ket{{\bf}}
\end{align*}

for every computational basis state $\ket{y}$ on the subsystem of dimension $2^d$.

## Representing Oracles in Python

I chose to represent oracles as a class. The important information is: 

- the number of working qubits
- the number of auxillary qubits
- the function
- the circuit
- the guaranteed relation between the function and circuit that defines the oracle type.

The abstract class ```oracle```  has subclasses for each oracle type that implement a method called ```.check_guarantee()``` that defines the oracle type and asserts that the circuit (which has not yet been implemented) does its job.

A particular oracle will be represented as a class that inherits from its corresponding oracle type class (e.g. ```PhaseOracle```). The particular oracle will need to implement a circuit for that oracle.

This project focuses on FunctionOracle and PhaseOracle, so I created additional classes to initialize these types of oracles from functions $f:\mathbb{N}\to \mathbb{R}$.

In [37]:
def my_function(x:int) -> float:
    """
    This example function demonstrates how we can initialize oracles based on functions.
    """
    return (x**0.5) %1

U=FunctionOracleFromFunction(2,my_function) #initialize a FunctionOracle on 2 qubits for my_function
O=PhaseOracleFromFunction(4,my_function) #initialize a PhaseOracle on 2 qubits for my_function
print(U.circuit().draw(output="text",fold=-1))
print(O.circuit().draw(output="text",fold=-1))

#checks that U and O satisfies
U.check_guarantee()
O.check_guarantee()

        ┌───┐              ┌───┐              ┌───┐              ┌───┐                   ┌───┐              ┌───┐              ┌───┐                    ┌───┐                     ┌───┐                           ┌───┐       ┌───┐                    ┌───┐                   
q202_0: ┤ X ├──────────────┤ X ├──────■───────┤ X ├──────■───────┤ X ├───────────────────┤ X ├──────■───────┤ X ├──────■───────┤ X ├────────────────────┤ X ├──────────■──────────┤ X ├─────────■─────────────────┤ X ├───────┤ X ├─────────■──────────┤ X ├─────────■─────────
        ├───┤              └─┬─┘      │       └─┬─┘      │       ├───┤┌───┐              └─┬─┘      │       └─┬─┘      │       ├───┤                    └─┬─┘          │          └─┬─┘         │                 └───┘       └─┬─┘         │          └─┬─┘         │         
q202_1: ┤ X ├──────■─────────■────────┼─────────■────────┼───────┤ X ├┤ X ├──────■─────────■────────┼─────────■────────┼───────┤ X ├─────────■────────────■────────────┼────────────■───

## From Function Oracles to Phase Oracles

We focus on the conversion from function oracles to phase oracles. Suppose that we have a function oracle $U_f$, and we want to construct the corresponding phase oracle, $O_f$. 

The naive way to achieve this is to use an intermediate step of a binary oracle, where $f(x)$ is computed and stored in a register. Then we can apply phase gates to retrieve the phase from the computational basis state. Because we use a binary oracle as an intermediate step, our resulting phase oracle can only be as accurate as the binary approximation that is stored in the $d$-qubit output register. Therefore, each additional bit of accuracy for the resulting phase oracle requires an additional axuillary qubit. The number of axuillary qubits required for this naive method is $\Theta(\log(\frac{1}{\epsilon}))$, where $\epsilon$ is the desired error.

There is a better conversion from function oracles to phase oracles that skips the intermediate step of converting to a binary oracle and requires fewer auxillary qubits to achieve the same accuracy.

Here is a recipe for the conversion from a ```FunctionOracle``` $U_f$ to a ```PhaseOracle``` $O_f$, following [Theorem 14](https://arxiv.org/pdf/1711.00465).

0. Define the grover operator $G$ as applying $U$, then flipping phases of basis states whose last qubit is in state 1, then applying $U^{-1}$, then flipping the phases of the basis states whose auxillary qubits are all $0$
1. Approximate the desired diagonal unitary for the phase oracle with a taylor expansion.
2. Truncate the taylor expansion after a few $M \in O (\frac{\log(\frac{1}{\epsilon})}{\log(\log(\frac{1}{\epsilon}))})$ terms.
3. Algebraically manipulate the taylor expansion to obtain a polynomial in the operator $G$.
4. Use the techinque of Linear Combinations of Unitaries to implement a circuit $V$ that uses $O(\log(M))$ auxillary qubits to implement the polynomial of $G$ using $O(\log(M))$ extra qubits.
5. Use Oblivious Amplitude Amplification to boost the probability of the previous step.

This technique uses only $\log(\log(\frac{1}{\epsilon}))$ auxillary qubits, and so represents an exponential space saving compared to the naive method.

### Linear Combination of Unitaries

[Source](https://arxiv.org/pdf/1511.02306), see Lemma 8.

A key step in the conversion from $U_f$ to $O_f$ is to approximate the desired phase oracle $O_f$ as a linear combination of powers of the operator $G$. Then, in Step $4$, we implement our approximation of $O_f$ as a linear combination of powers of $G$.

When an operator (not necessarily unitary) $L$ is a linear combination of unitary matrices (in our case, powers of $G$), we can simulate the behavior $\ket{\psi}\mapsto \frac{L\ket{\psi}}{\sqrt{\braket{\psi|\psi}}}$ by creating an auxillary register that is used to decide which unitary matrix to apply. We initialize this auxillary register so that its probabilities correspond to the coefficients in the linear combination. The axuillary register controls which unitary matrix to apply.

We obtain the desired linear combination of unitaries by applying the circuit to a state $\ket{\psi}\ket{{\bf 0}}$ and measuring the auxillary qubits. If the outcome is ${\bf 0}$, then the working qubits will collapse into the desired state.

In our implementation, we assume that the unitary matrices are all powers of a specific matrix, $Y$, where we will set $Y=G$ and $L=O_f$ in the larger conversion procedure. This simplification allows us to use the n-controlled G pattern, where the $i^{th}$ axuillary qubit is used to apply a controlled $G^{2^i}$ gate to the working register.

The reference assumes that the coefficients are all nonnegative real numbers. We cannot make this assumption in our implementation because the coefficients $\beta_m$ of the expansion of $O_f$ in terms of $G$ are complex numbers. We handle these complex coefficients by working with their magnitudes, then adding phase gates to the auxillary qubits to restore the phases. These phases jump to the working qubits, and can be thought of as phases for the n-controlled $Y$ gates.

The reference also assumes that we are applying a linear combination of $Y^0,Y^1,Y^2,\dots,Y^z$, for some $z\in \mathbb{N}$. We actually need a linear combination of $G^{-M},G^{-M+1},\dots,G^{M-1},G^{M}$, $O_f\approx \sum_{m=-M}^M \beta_m G^{M}$. To account for this, we can create the circuit for the linear combination $G^{M} O_f \approx \sum_{m=0}^{2M} \beta_{m-M} G^m$ and introduce a (uncontrolled) $G^{-M}$ gate applied to the working register.

In [36]:
matrix = random_unitary(4, seed=42).data
 #Here is a unitary matrix.
print("random 2 qubit matrix:", matrix)

coefficients = [2,3,5] #We'll combine the matrix with these coefficients.
working_qubits = QuantumRegister(2)
Y=QuantumCircuit(np.log2(len(matrix)),name="Y")
Y.unitary(matrix, range(Y.num_qubits))
Y.to_gate()
Y.name = "Y"

circuit = linear_combination_of_unitaries_circuit(coefficients,Y)
print(circuit.draw(output="text",fold=-1))

initial_state = Statevector.from_int(2,dims=2**(circuit.num_qubits)) #set the initial state to 0010
initial_state_working_register = Statevector.from_int(2,dims=2**(Y.num_qubits))
state = initial_state.evolve(circuit)

linear_combination_of_unitaries = np.array(sum(coefficients[i] * np.linalg.matrix_power(Operator(Y).data, i) for i in range(len(coefficients))))


ideal_state_data = linear_combination_of_unitaries @ initial_state_working_register.data
ideal_state_data = ideal_state_data/np.linalg.norm(ideal_state_data)
while True:
    outcome, new_state = state.measure([2,3]) #Measure the last two qubits,
    if outcome == "00":
        marginal_state = new_state.data[:2**Y.num_qubits]
        assert np.allclose(marginal_state,ideal_state_data)
        print("The two vectors are close:", marginal_state, ideal_state_data, sep="\n")
        break

random 2 qubit matrix: [[ 0.14480819+0.1752384j  -0.51892816-0.52424259j -0.14955858+0.312755j
   0.16913481-0.50538631j]
 [-0.92717439-0.08785062j -0.11260331-0.1818585j   0.12255872+0.09640286j
  -0.24498509-0.05045841j]
 [-0.00798428-0.20355071j -0.38932055-0.05180925j  0.26051706+0.32864025j
   0.44517308+0.65589332j]
 [ 0.03137922+0.19613952j  0.4980475 +0.08846049j  0.34078865+0.750661j
   0.01464807-0.15755843j]]
                                     ┌───────┐┌───────┐            
target register_0: ──────────────────┤0      ├┤0      ├────────────
                                     │  Y**1 ││  Y**2 │            
target register_1: ──────────────────┤1      ├┤1      ├────────────
                   ┌───────┐┌───────┐└───┬───┘└───┬───┘┌──────────┐
   aux register_0: ┤0      ├┤0      ├────■────────┼────┤0         ├
                   │  Prep ││  Diag │             │    │  Prep_dg │
   aux register_1: ┤1      ├┤1      ├─────────────■────┤1         ├
                   └───────┘└───

### Oblivious Amplitude Amplification

We refer to [Source](https://arxiv.org/pdf/1711.00465) (just above Equation 15) for coefficients of $\beta_m$ that we need in order to approximate $O_f$ as a linear combination $\sum_m \beta_m G^m$. With these value, our linear combination of unitaries circuit almost achieves the goal of approximating $O_f$. However, we must post-select on measuring the auxillary qubits to be $0$ and repeat the circuit when they are not. This repeat-until-success loop is a classical procedure, and it is not clear how to represent it as a quantum circuit. The technique of oblivious amplitude amplification solves the problem by boosting the probability of success to near-certainty. In our case, it consists of applying a certain Grover-like operator twice.

In [35]:
def f(x):
    return x**1.5 %1


Uf = FunctionOracleFromFunction(2,f)
Uf.check_guarantee()
print(f"Function Oracle:" )
print(Uf.circuit().draw(output="text", fold=-1))
Of=function_oracle_to_phase_oracle(Uf)
Of.check_guarantee()
print("Phase Oracle:")
print(Of.circuit().draw(output="text",fold=-1))

Function Oracle:
        ┌───┐              ┌───┐              ┌───┐              ┌───┐                   ┌───┐              ┌───┐              ┌───┐                   ┌───┐                    ┌───┐                          ┌───┐        ┌───┐                     ┌───┐                    
q193_0: ┤ X ├──────────────┤ X ├──────■───────┤ X ├──────■───────┤ X ├───────────────────┤ X ├──────■───────┤ X ├──────■───────┤ X ├───────────────────┤ X ├─────────■──────────┤ X ├─────────■────────────────┤ X ├────────┤ X ├──────────■──────────┤ X ├─────────■──────────
        ├───┤              └─┬─┘      │       └─┬─┘      │       ├───┤┌───┐              └─┬─┘      │       └─┬─┘      │       ├───┤                   └─┬─┘         │          └─┬─┘         │                └───┘        └─┬─┘          │          └─┬─┘         │          
q193_1: ┤ X ├──────■─────────■────────┼─────────■────────┼───────┤ X ├┤ X ├──────■─────────■────────┼─────────■────────┼───────┤ X ├─────────■───────────■───────────┼─

### Analysis

We already gave a brief analysis that the circuit uses $\log(\log(\frac{1}{\epsilon}))$ ancillas, where we are constructing a unitary that is $\epsilon$ close in operator norm to an exact phase oracle, which is an exponential space saving over the naive conversion.

Another measure of complexity for our conversion is the number of calls made to $O_f$. We can see that the Linear Combination of Unitaries circuit uses $\sum_{i=0}^a 2^a \approx \log(\frac{1}{\epsilon})$ calls to $U_f$, where $a$ is the number of auxillary qubits. The two Oblivious Grover circuits each make two calls to the Linear Combinations of Unitaries circuit, but we ignore this constant factor in our rough analysis.